# LAB 3: Natural Language Generation

In [1]:
!git clone https://github.com/elenipapadopulos/NLP_LAB3_Datasets.git

Cloning into 'NLP_LAB3_Datasets'...
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 11 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (11/11), 6.23 KiB | 3.12 MiB/s, done.
Resolving deltas: 100% (3/3), done.


In [2]:
!pip install transformers

## Exercises

### Ex. 1: build a trigram model with backoff

In this exercise we are going to build a trigram model, implementing a back-off strategy.

Let's compute the unigram and bigram models.

In [3]:
import nltk
nltk.download('punkt_tab')
nltk.download('brown') # import the Brown corpus from NLTK
nltk.download('punkt') # import tokenizer
import math

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [4]:
from nltk.corpus import brown

sentences = brown.sents()[:10000]
tokenized_corpus = []

for sentence in sentences:
  tokens = nltk.word_tokenize(' '.join(sentence))

  tokens = ['<s>'] + tokens + ['</s>']

  tokenized_corpus.extend(tokens)


In [5]:
from collections import Counter, defaultdict
from nltk import ngrams

unigram = list(ngrams(tokenized_corpus, 1))

unigram_count = Counter(unigram) # computes the occurrences

**Ex.1.1** Let's start by computing the unigram distribution: we already have the unigram counts, so what's left to do is normalizing them.

In [6]:
## Ex 1.1

## compute the total number of tokens
num_tokens = sum(unigram_count.values())


## create a unigram model dictionary
## key: word
## value: probability
unigram_model = defaultdict(lambda: defaultdict(lambda: 0))
for unigram, count in unigram_count.items():
  unigram_model[unigram[0]] = count / num_tokens

w1 = 'there'
print(f"{w1} -> {unigram_model[w1]}")

there -> 0.0013045560381128524


Compute the bigram model in the following cell.

In [7]:
### Compute bigrams
from nltk import bigrams

bigram = list(bigrams(tokenized_corpus))

# build a bigram model (dictionary of counts values)
bigram_model = defaultdict(lambda: defaultdict(lambda: 0))


for w1, w2 in bigram:
    bigram_model[w1][w2] += 1

for w1 in bigram_model:
    total_count = float(sum(bigram_model[w1].values()))
    for w2 in bigram_model[w1]:
        bigram_model[w1][w2] /= total_count

As we are working with trigrams, it is important to make sure that have two `<s>` at the beginning of the sentence to be able to estimate the first word.

In [8]:
tokenized_corpus_tri = []
for sentence in sentences:

  tokens = nltk.word_tokenize(' '.join(sentence))

  tokens = ['<s>', '<s>'] + tokens + ['</s>']

  tokenized_corpus_tri.extend(tokens)

We can compute now our trigrams, using the `trigram` function.

In [9]:
from nltk import trigrams

trigram = list(trigrams(tokenized_corpus_tri))

# let's define the trigram model as a dictionary of dictionaries
trigram_model = defaultdict(lambda: defaultdict(lambda: 0))

**Ex 1.2.**: Compute co-occurrences counts.
Note that the dictionary `trigram_model` takes as input a tuple of words as you can see from the example in the cell.

In [10]:
## Ex 1.2

## compute trigram counts
for w1, w2, w3 in trigram:
    trigram_model[(w1, w2)][w3] += 1

w1_w2 = ('there', 'were')
print(f"{w1_w2} -> {dict(trigram_model[w1_w2])}")

('there', 'were') -> {'no': 2, 'lots': 1, 'several': 1, 'such': 2, 'not': 2, 'only': 1, "n't": 1, 'plenty': 1, 'as': 1, ',': 1, 'other': 1, 'in': 2, 'those': 3, 'moderates': 1, 'to': 2, 'a': 1, 'actually': 1, 'one': 1, 'solid': 1, 'evidences': 1, 'flashes': 2, 'very': 1}


**Ex 1.3** Compute trigram probabilities by normalizing counts.

In [11]:
## Ex 1.3

## compute trigram probabilities
for w1_w2 in trigram_model:
    total_count = float(sum(trigram_model[w1_w2].values()))
    for w3 in trigram_model[w1_w2]:
        trigram_model[w1_w2][w3] /= total_count

w1_w2 = ('there', 'were')
print(f"{w1_w2} -> {dict(trigram_model[w1_w2])}")

('there', 'were') -> {'no': 0.06666666666666667, 'lots': 0.03333333333333333, 'several': 0.03333333333333333, 'such': 0.06666666666666667, 'not': 0.06666666666666667, 'only': 0.03333333333333333, "n't": 0.03333333333333333, 'plenty': 0.03333333333333333, 'as': 0.03333333333333333, ',': 0.03333333333333333, 'other': 0.03333333333333333, 'in': 0.06666666666666667, 'those': 0.1, 'moderates': 0.03333333333333333, 'to': 0.06666666666666667, 'a': 0.03333333333333333, 'actually': 0.03333333333333333, 'one': 0.03333333333333333, 'solid': 0.03333333333333333, 'evidences': 0.03333333333333333, 'flashes': 0.06666666666666667, 'very': 0.03333333333333333}


**Ex. 1.4** Implement the back-off function.

The function backoff should take as input the trigram, bigram and unigram models, the context words $w_1$ and $w_2$ and a discount factor $\alpha$ and it should return the next word $w_3$ to be generated and its probabilty.


Recall: for each $w_3$, if $P(w_3 \mid w_1, w_2)$ is not in the trigram model, then use $P(w_3 \mid w_2)$ (discounted by $\alpha$). If there is no $P(w_3 \mid w_2)$, then use $P(w_3)$ (discounted by $\alpha^2$).


In [12]:
def backoff(trigram_model, bigram_model, unigram_model, w1, w2, alpha):

  # remove the <s> token from the unigram model before using it for backoff:
  # it is the most probable token and will be chosen if algorithm backs off to unigrams

  # given the context w1_w2, use backoff to find word with the highest probability:
  # for every w in vocabulary, check if p(w|w1_w2) exists, otherwise back off to bigrams
  # if there is no p(w|w2), then back off to unigram probability

  # use the computed trigram, bigram and unigram models

  # greedy decoding: you should return next_word, probability

  unigram_model.pop('<s>', None)
  next_word = None
  probability = -1.0

  if (w1, w2) in trigram_model:
    probability = max(trigram_model[(w1, w2)].values())
    next_word = max(trigram_model[(w1, w2)], key=trigram_model[(w1, w2)].get)
  elif w2 in bigram_model:
    probability = (alpha) * max(bigram_model[w2].values())
    next_word = max(bigram_model[w2], key=bigram_model[w2].get)
  else:
    probability = (alpha ** 2) * max(unigram_model.values())
    next_word = max(unigram_model, key=unigram_model.get)

  return next_word, probability


The function can be used now during generation, as you can see in the following cell.

In [13]:
def backoff_generation(trigram_model, bigram_model, unigram_model, seed_words, alpha, max_lenght=15):

  w1, w2 = seed_words
  generated_words = [w1, w2]

  log_prob_sum = 0.0
  N = 0

  for _ in range(max_lenght):

    next_word, probability = backoff(trigram_model, bigram_model, unigram_model, w1, w2, alpha)

    generated_words.append(next_word)
    log_prob_sum += math.log(probability)
    N += 1

    w1, w2 = w2, next_word

  perplexity = math.exp(-log_prob_sum / N) if N > 0 else float('inf')

  return ' '.join(generated_words), perplexity

In [14]:
seed = ("well", "if")
alpha = 0.4
generated_sentence, perplexity = backoff_generation(trigram_model, bigram_model, unigram_model, seed, alpha)
print("Generated sentence:", generated_sentence)
print("Perplexity:", perplexity)

Generated sentence: well if the present city CD head , believes . </s> <s> <s> The President said he
Perplexity: 3.555641997731578


In [15]:
seed = ("there", "was")
alpha = 0.4
generated_sentence, perplexity = backoff_generation(trigram_model, bigram_model, unigram_model, seed, alpha)
print("Generated sentence:", generated_sentence)
print("Perplexity:", perplexity)

Generated sentence: there was no problem . </s> <s> <s> The President said he was the first time in
Perplexity: 4.456818195206422


### Ex. 2: typo detection

In [16]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import matplotlib.pyplot as plt
import math
import pandas as pd

In this exercise, you are asked to use a Large Language Model (GPT2) to detect typos in sentences.

Let's start by uploading our dataset: it is a collection of 52 sentences of various type, half of which are grammatically sound.

In [17]:
df = pd.read_csv("/content/NLP_LAB3_Datasets/typo_dataset1.csv")

You can see that these sentences contain typos (so spelling errors or missing letters) but also grammatical errors like omophones.

In [18]:
print(f"Text: {df['text'][26]} \nLabel: {df['label'][26]} (Typo)")

Text: <s> If the owewr of the vehicle is not licensed to drive, the owner's license to drive mya be suspended. 
Label: 0 (Typo)


In [19]:
print(f"Text: {df['text'][21]} \nLabel: {df['label'][21]} (Correct)")

Text: <s> I finally submitted my project 💻 after working on it for days, and now I’m treating myself to some ice cream. 
Label: 1 (Correct)


Let's import GPT2.

In [ ]:
device =  "cuda:0" if torch.cuda.is_available() else "cpu"

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

We will use the token \<s> as our marker for the beginning of a sentence. First, we need to add it to the tokenizer’s vocabulary, then set it as the bos_token, and finally, resize the model’s embedding size.


In [21]:
bos_token = "<s>"
tokenizer.add_tokens([bos_token])
tokenizer.bos_token = bos_token
model.resize_token_embeddings(len(tokenizer))


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(50258, 768)

**Ex 2.1** Write a function that returns the log-probability assigned to each generated token.

You can use the `get_next_word_probs` function as a reference, but remember that this time we’re focusing on the distribution of tokens **within** the sentence, rather than the probability distribution of the next token. You can follow the comments we left in the box to guide you in the implementation.


In [22]:
def get_token_logprobs(sentence):

    ## tokenize the sentence, compute the output and retrieve logits
    batch = tokenizer([sentence], padding=False, truncation=False, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model(**batch, output_hidden_states=True)
        logits = output.logits

    ## remember: logits.shape is (1, sen_len, model_size])
    ## remove the logits relative to the last token: we are not interested in next token generation
    ## expected shape: (sen_len, model_size) (suggestion: use squeeze))
    logits = logits.squeeze()

    ## retrieve the indices (input_ids) of the sentence
    ## hint: remove the input_id relative to the bos
    ## expected shape: (sen_len)
    indices = batch.input_ids.squeeze()[1:]

    ## compute log-probabilities
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

    ## retrieve the probabilities of the tokens
    token_logprobs = log_probs[range(len(indices)), indices]

    ## convert input ids to tokens to obtain a list of tokens
    tokens = tokenizer.convert_ids_to_tokens(indices)

    return tokens, token_logprobs.tolist()

**Ex 2.2** Write a function that returns the cumulative log-probability **up to each token** in a sentence, using the probabilities computed before.

Remind that, as we are considering log-probabilities, you should **sum** the individual log-probabilities of each individual token.

Hint: you could return a list of elements like (w, cumulative_probability_up_to_w)


In [23]:
def get_cumulative_token_logprobs(sentence):

    tokens, token_logprobs = get_token_logprobs(sentence)

    # write your code here
    cumulative_probability = 0.0
    cumulative_logprobs = []
    for token, token_logprob in zip(tokens, token_logprobs):
        cumulative_probability += token_logprob
        cumulative_logprobs.append((token, cumulative_probability))

    return cumulative_logprobs

**Ex. 2.3** Write a function that determines whether a sentence contains typos or not based on the difference of log-probabilities of consecutive tokens/words. The hypothesis is that a significant drop in probability between consecutive tokens may indicate that the model did not expect that token, potentially signaling a typo.

To implement this, we can define a threshold: if the difference between the log-probabilities of consecutive tokens exceeds this threshold, we can assume the sentence likely contains a typo.

The function should take as input the sentence to be analyzed and the threshold to apply for detection and it should return 0 if the sentence is flagged as potentially containing typos or	1 if the sentence is considered correct.

In the function you should confront consecutive log-probabilities and check whether, for at least one pair, their difference is above the set threshold: in that case, classify the whole sentence as incorrect.

In [24]:
def detect_typos(sentence, threshold=5):

    word_logprobs = get_cumulative_token_logprobs(sentence)

    # write your code here
    label = 1
    for i in range(1, len(word_logprobs)):
      if (abs(word_logprobs[i][1] - word_logprobs[i-1][1])) > threshold:
        label = 0
        break

    return label

**Ex 2.4** Test your function experimenting with different thresholds.
Select the best threshold on the whole data and report the accuracy in the Moodle.

You should achieve an accuracy higher than 0.70.

In [25]:
correct = 0

for i, row in df.iterrows():
  sentence, label = row['text'], row['label']
  pred = detect_typos(sentence, threshold=9)

  if label == pred:
    correct += 1

print(f"Accuracy: {correct/len(df)}")

Accuracy: 0.75


**Error Analysis**: in this exercise, we exploited the probability distribution of a Large Language Model to detect typos.
Did you notice any linguistic or grammatical feature that make detection more accurate? Do you think relying solely on threshold-based differences in log-probabilities is sufficient or should we use a more sophisticated and complete system? Write your comments in the box below.

In some sentences of the dataset there are some emojis, this will lower the accuracy of the threshold-based method that we implemented. The problem is that the probability of having an emoji after an english word is extremely low since GPT-2 is trained on standard English text. As a consequence, an emoji causes a massive drop in the sequence's log-probability, easily exceeding our set threshold. This causes our implementation to incorrectly label sentences with no typos as ones with typos, which will bring down the accuracy. Furthermore, relying solely on threshold-based differences in left-to-right log-probabilities is not sufficient. A more sophisticated system using a bidirectional model like BERT would be highly beneficial for this task. This is because BERT can analyse both the left and right context of a word simultaneously, so it has a much stronger understanding of the full sentence structure. Therefore instead of relying on an arbitrary probability drop, a model like BERT could be explicitly fine-tuned for token classification to identify typos.

## Moodle submission

Extract the code to run exercise 1 and 2 from the notebook as a .py file (you might have to add some initial import instructions). Make sure the code runs and computes the accuracy correctly. Upload the code on the activity "Lab assignment 3" on the Moodle of the course. In the comment box on Moodle report your accuracy and your comments related to the error analys (box right above)